# CLR

In [1]:
# ==============================================================================
# COMPOSITIONAL DATA ANALYSIS (CLR) PIPELINE
# ==============================================================================

options(warn = -1)
options(stringsAsFactors = FALSE)
options(future.globals.maxSize = 10 * 1024^3)

quiet_library <- function(pkg) {
  suppressWarnings(suppressMessages(require(pkg, character.only = TRUE)))
}

pkgs <- c("dplyr", "zCompositions", "compositions", "reshape2", "ggplot2",
          "ggpubr", "tidyr", "tibble", "stringr", "purrr", "rlang")
invisible(lapply(pkgs, quiet_library))

# ==============================================================================
# 1. MASTER CONFIGURATION
# ==============================================================================
base_dir      <- "/data/ag_reichart_lindberg/nhadizad/nastaran/projects/xeno/analysis/v1/CLR/v1/cell_state/ref_3month"
metadata_name <- "metadata_for_clr_cellstate.csv"

cell_types_to_run <- c("CM", "FB", "EC", "Mye_papio", "Lymph_papio", "Mye_sus", "Lymph_sus")

sample_col    <- "Sample_ID"
target_col    <- "cell_state"
condition_col <- "Condition"

ref_condition       <- "3month-protocol"
pseudocount_value   <- 0.2
min_cells_threshold <- 5

condition_order <- c("Rejection", "PCMV/PRV", "Prematurely-terminated",
                     "6month-protocol", "3month-protocol", "Control")

custom_theme <- theme(
  text = element_text(size = 5, face = "bold", family = "sans", color = "black"),
  axis.text.x = element_text(angle = 90, hjust = 1, face = "bold", size = 5, family = "sans", color = "black"),
  axis.text.y = element_text(face = "bold", size = 5, family = "sans", color = "black"),
  axis.title = element_text(face = "bold", size = 5, family = "sans", color = "black"),
  legend.text = element_text(face = "bold", size = 5, family = "sans", color = "black"),
  legend.title = element_text(face = "bold", size = 5, family = "sans", color = "black"),
  panel.background = element_blank(),
  panel.grid.major = element_blank(),
  panel.grid.minor = element_blank()
)

our_fill_scale_gradient2 <- scale_fill_gradient2(
  limits = c(-1, 1), oob = scales::squish, high = "#FF6600", mid = "white", low = "#6699FF"
)

# ==============================================================================
# 2. INITIALISE MASTER COLLECTOR  <<<< NEW
# ==============================================================================
master_results_list <- list()

# ==============================================================================
# 3. START MAIN LOOP
# ==============================================================================

for (ct in cell_types_to_run) {
  cat("\n=========================================================\n")
  cat("PROCESSING CELL TYPE:", ct, "\n")
  cat("=========================================================\n")

  work_dir <- file.path(base_dir, ct)
  if (!dir.exists(work_dir)) next
  setwd(work_dir)
  dir.create("results", showWarnings = FALSE)

  meta <- read.csv(metadata_name, check.names = FALSE)
  if (colnames(meta)[1] == "") { meta <- meta[, -1] }

  # --- FIX: Ensure no ghost factor levels ---
  meta$cell_state <- as.character(meta$cell_state)
  cat("DEBUG - Unique states in", ct, ":", paste(unique(meta$cell_state), collapse=", "), "\n")

  prop.coarse <- meta %>%
    group_by(!!sym(target_col), !!sym(sample_col), !!sym(condition_col)) %>%
    summarise(count = n(), .groups = "drop")

  ct_mat <- reshape2::acast(
    prop.coarse,
    as.formula(paste(sample_col, target_col, sep = "~")),
    value.var = "count", fun.aggregate = mean, fill = 0
  )

  cat("Debug:", ct, "has", ncol(ct_mat), "states and", nrow(ct_mat), "samples.\n")

  if (ncol(ct_mat) < 2) {
    cat("SKIPPING:", ct, "- CLR requires >= 2 states.\n")
    next
  }

  # APPLY PSEUDOCOUNT
  ct_mat <- ct_mat + pseudocount_value
  prop_matrix <- sweep(ct_mat, 1, rowSums(ct_mat), "/")

  prop_df <- as.data.frame(prop_matrix) %>%
    tibble::rownames_to_column(sample_col) %>%
    left_join(distinct(meta, !!sym(sample_col), !!sym(condition_col)), by = sample_col)
  prop_df[[condition_col]] <- trimws(prop_df[[condition_col]])

  # Statistical Function  — UNCHANGED
  da.by.covariate <- function(ct_in, target_name, condition_vec) {
    Z <- compositions::clr(compositions::acomp(ct_in))
    data <- data.frame(abundance = Z[, target_name], proportion = ct_in[, target_name], Condition = condition_vec)
    m  <- lm(abundance ~ Condition, data = data)
    smr  <- t(coef(summary(m))[2, ])
    m2 <- lm(proportion ~ Condition, data = data)
    smr2 <- t(coef(summary(m2))[2, ])
    return(data.frame(smr, smr2, mean.prop = mean(data$proportion, na.rm = TRUE),
                      mean.prop.ctrl = coef(m2)[1], check.names = FALSE))
  }

  conditions_to_test <- setdiff(unique(prop_df[[condition_col]]), ref_condition)
  all_results <- list()

  for (cond in conditions_to_test) {
    selected <- prop_df %>%
      filter(!!sym(condition_col) %in% c(ref_condition, cond)) %>%
      filter(!!sym(sample_col) %in% rownames(ct_mat)[rowSums(ct_mat) >= min_cells_threshold])

    if (sum(selected[[condition_col]] == ref_condition) > 0 &&
        sum(selected[[condition_col]] == cond) > 0) {

      selected$Condition_Factor <- factor(selected[[condition_col]], levels = c(ref_condition, cond))
      valid_targets <- setdiff(colnames(selected), c(sample_col, condition_col, "Condition_Factor"))

      results <- lapply(valid_targets, function(t)
        da.by.covariate(selected[, valid_targets, drop = FALSE], t, selected$Condition_Factor))

      res <- do.call(rbind, results)
      rownames(res) <- valid_targets
      all_results[[cond]] <- res
    }
  }

  if (length(all_results) == 0) next

  # Build per-cell-type results_df  — column rename Target -> cell_state ONLY here
  results_df <- imap_dfr(all_results, ~ data.frame(
    Condition      = .y,
    Target         = rownames(.x),          # keep as "Target" internally for plots
    PValue         = .x[, "Pr(>|t|)"],
    EffectSize     = .x[, "Estimate"],
    MeanPropControl= .x[, "mean.prop.ctrl"],
    stringsAsFactors = FALSE
  )) %>%
    group_by(Condition) %>%
    mutate(FDR = p.adjust(PValue, method = "BH")) %>%
    ungroup()

  # ------------------------------------------------------------------
  # FOLD CHANGE & LOG2FC CALCULATION                          
  #
  # For each cell_state x Condition pair:
  #   mean_prop_cond  = mean proportion of that cell_state across
  #                     samples belonging to that Condition
  #   mean_prop_ref   = mean proportion of that cell_state across
  #                     samples belonging to ref_condition
  #   fold_change     = mean_prop_cond / mean_prop_ref
  #   log2FC          = log2(fold_change)
  # ------------------------------------------------------------------
  prop_long_full <- as.data.frame(prop_matrix) %>%
    tibble::rownames_to_column(sample_col) %>%
    tidyr::pivot_longer(-!!sym(sample_col), names_to = "Target", values_to = "Prop") %>%
    dplyr::left_join(dplyr::distinct(meta, !!sym(sample_col), !!sym(condition_col)), by = sample_col)

  # Mean proportion of ref condition per cell_state
  ref_mean_props <- prop_long_full %>%
    dplyr::filter(!!sym(condition_col) == ref_condition) %>%
    dplyr::group_by(Target) %>%
    dplyr::summarise(mean_prop_ref = mean(Prop, na.rm = TRUE), .groups = "drop")

  # Mean proportion of each non-ref condition per cell_state
  cond_mean_props <- prop_long_full %>%
    dplyr::filter(!!sym(condition_col) != ref_condition) %>%
    dplyr::group_by(!!sym(condition_col), Target) %>%
    dplyr::summarise(mean_prop_cond = mean(Prop, na.rm = TRUE), .groups = "drop") %>%
    dplyr::rename(Condition = !!sym(condition_col))

  fc_df <- cond_mean_props %>%
    dplyr::left_join(ref_mean_props, by = "Target") %>%
    dplyr::mutate(
      fold_change = mean_prop_cond / mean_prop_ref,
      log2FC      = log2(fold_change)
    ) %>%
    dplyr::select(Condition, Target, fold_change, log2FC)

  # Join fold-change columns into results_df
  results_df <- results_df %>%
    dplyr::left_join(fc_df, by = c("Condition", "Target"))

  # ------------------------------------------------------------------
  # Save per-cell-type CSV  (Target kept as label for backward compat)
  # ------------------------------------------------------------------
  write.csv(results_df, file.path("results", paste0("CLR_CellState_", ct, "_Stats.csv")),
            row.names = FALSE)

  # ------------------------------------------------------------------
  # Build master-ready slice: rename + add metadata columns 
  # ------------------------------------------------------------------
  master_slice <- results_df %>%
    dplyr::rename(cell_state = Target) %>%
    dplyr::mutate(
      Ref_condition = ref_condition,   # always "3month-protocol"
      cell_type     = ct
    ) %>%
    dplyr::select(
      Condition, Ref_condition, cell_type, cell_state,
      PValue, FDR, EffectSize, MeanPropControl,
      fold_change, log2FC
    )

  master_results_list[[ct]] <- master_slice

  # ==================================================================
  # VISUALISATION  — completely unchanged, still uses "Target" column
  # ==================================================================
  results_df$Condition <- factor(results_df$Condition, levels = condition_order)

  p1 <- ggplot(results_df, aes(x = Target, y = Condition, fill = EffectSize)) +
    geom_tile(color = "white", linewidth = 0.15) + our_fill_scale_gradient2 +
    labs(y = "Condition", x = target_col, fill = "Proportion change\nrelative to Reference") +
    custom_theme + theme(legend.position = "bottom") +
    geom_text(data = subset(results_df, FDR < 0.05),
              aes(label = gsub("e-0", "e-", formatC(PValue, format = "e", digits = 1))),
              color = "black", size = 5 / .pt, fontface = "bold", family = "sans")

  ggsave(file.path("results", paste0("CLR_CellState_", ct, "_Heatmap.pdf")),
         plot = p1, width = 3.5, height = 2.5, units = "in", dpi = 400)

  prop_long <- as.data.frame(prop_matrix) %>%
    tibble::rownames_to_column(sample_col) %>%
    tidyr::pivot_longer(-!!sym(sample_col), names_to = "Target", values_to = "Prop")

  ref_means <- prop_long %>%
    dplyr::left_join(dplyr::distinct(meta, !!sym(sample_col), !!sym(condition_col)), by = sample_col) %>%
    dplyr::filter(!!sym(condition_col) == ref_condition) %>%
    dplyr::group_by(Target) %>%
    dplyr::summarise(MeanProp = mean(Prop, na.rm = TRUE), .groups = "drop")

  p_ref <- ggplot(ref_means, aes(x = Target, y = MeanProp, fill = Target)) +
    geom_col(color = "black", alpha = 0.8) +
    geom_text(aes(label = scales::percent(MeanProp, accuracy = 0.1)),
              vjust = -0.5, size = 5 / .pt, fontface = "bold", family = "sans", color = "black") +
    labs(y = paste("Mean % in", ref_condition), x = target_col) +
    theme_bw() + custom_theme +
    theme(axis.title.x = element_blank(), legend.position = "none") +
    scale_y_continuous(labels = scales::percent_format(accuracy = 1),
                       expand = expansion(mult = c(0, 0.15)))

  ggsave(file.path("results", paste0("CLR_CellState_", ct, "_Barplot.pdf")),
         plot = p_ref, width = 3.5, height = 2.5, units = "in", dpi = 400)

  p_ref_no_x <- p_ref + theme(axis.title.x = element_blank(),
                               axis.text.x  = element_blank(),
                               axis.ticks.x = element_blank())

  plot_all <- ggarrange(p_ref_no_x, p1, ncol = 1, nrow = 2,
                        align = "v", heights = c(2.25, 3))

  ggsave(file.path("results", paste0("CLR_CellState_", ct, "_Master.pdf")),
         plot = plot_all, width = 3.5, height = 4, units = "in", dpi = 400)

  cat("Completed", ct, "\n")
}

# ==============================================================================
# 4. WRITE WHOLE-MASTER CSV  (after all cell types are done)      <<<< NEW
# ==============================================================================
if (length(master_results_list) > 0) {

  whole_master <- dplyr::bind_rows(master_results_list)

  master_out <- file.path(base_dir, "CLR_CellState_WholeMaster.csv")
  write.csv(whole_master, master_out, row.names = FALSE)

  cat("\n=========================================================\n")
  cat("WHOLE-MASTER CSV SAVED TO:\n", master_out, "\n")
  cat("Rows:", nrow(whole_master), " | Cell types:", length(master_results_list), "\n")
  cat("=========================================================\n")
}


PROCESSING CELL TYPE: CM 
DEBUG - Unique states in CM : CM3, CM2, Hypoxia_CM, CM4, ISG_CM, CM1, CM5 
Debug: CM has 7 states and 26 samples.
Completed CM 

PROCESSING CELL TYPE: FB 
DEBUG - Unique states in FB : Progenitor_FB, Homeo_FB, ECM_FB, Myo_FB, Hypoxia_FB, ISG_FB, Angio_FB, Prolif_Myo_FB 
Debug: FB has 8 states and 26 samples.
Completed FB 

PROCESSING CELL TYPE: EC 
DEBUG - Unique states in EC : Lymph_EC, Ven_EC, Arterial_EC, Cap_EC, Hypoxia_EC, Meso_EC, ISG_cap_EC, Prolif_EC, PCMV/PRV_EC 
Debug: EC has 9 states and 26 samples.
Completed EC 

PROCESSING CELL TYPE: Mye_papio 
DEBUG - Unique states in Mye_papio : DC_papio, MO_papio, ISG_MP_papio, LYVE1_MP_papio, AP_MP_papio, GBE1_MP_papio, SPP1_MP_papio, Groundstate_MP_papio, Prolif_Mye_papio 
Debug: Mye_papio has 9 states and 22 samples.
Completed Mye_papio 

PROCESSING CELL TYPE: Lymph_papio 
DEBUG - Unique states in Lymph_papio : CM_CD4T_papio, EM_CD8T_papio, Reg_CD4T_papio, Naive_CD4T_papio, Prolif_Lymph_papio, CD56_NK_papio